# MindRL Challenge: Cognitive Modeling Workbook

A standalone, self-contained walkthrough of the Bayes'd Misfits models.
Every piece of code that matters is in this notebook -- no external imports
from our project package. Run this on Colab (GPU recommended) or locally.

**What you will learn:**
1. How the Rescorla-Wagner learning model works, line by line
2. How dual-alpha learning rates capture asymmetric learning from good/bad news
3. How choice stickiness (perseveration) explains why people repeat choices
4. How HSSM fits these models to human data using Bayesian sampling
5. How we compare models on held-out data to pick the best one

**The task:** Humans play a 4-armed bandit (4 slot machines). Each trial they
pick one arm and get a reward. The reward rates slowly drift over time
(restless bandit). We model how humans learn which arm is best.


## 1. Setup

Install HSSM and import everything we need. On Colab this takes about 2 minutes.
If running locally with `uv sync` already done, the pip install is a no-op.


In [ ]:
import subprocess, sys
try:
    import hssm
except ImportError:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                     'git+https://github.com/lnccbrown/HSSM.git@main',
                     'arviz', 'pyhgf', 'pyarrow', 'huggingface_hub',
                     'scipy', 'pandas'], check=True)
    import hssm

import warnings, logging
warnings.filterwarnings('ignore')
logging.getLogger('pytensor').setLevel('ERROR')
logging.getLogger('jax._src.xla_bridge').setLevel('ERROR')

import numpy as np
import pandas as pd
import jax
import jax.numpy as jnp
from scipy.special import expit, logsumexp, logit

hssm.set_floatX('float32', update_jax=True)
print(f'JAX devices: {jax.devices()}')
print('Setup complete.')


**What just happened:**
- `hssm` is the Hierarchical Sequential Sampling Models library. It fits cognitive
  models to behavioral data using Bayesian statistics (MCMC sampling).
- `jax` is Google's numerical computing library for fast, differentiable computations.
- `float32` is set for speed (float64 broke the sampler for some models).
- The JAX devices line tells you if a GPU is available.


## 2. The Data: 4-Armed Bandit

The MindRL Challenge provides human behavioral data from a 4-armed bandit task.
Each trajectory is one independent 120-trial task run; one person can contribute multiple trajectories. We convert JSONL to a DataFrame.

**Key columns:**
- `response`: which arm the person chose (0, 1, 2, or 3)
- `feedback`: the reward, normalized to 0-1 (raw rewards are 1-100)
- `rt`: response time in seconds (or -1.0 if missing)
- `participant_id`: HSSM sequence key (one trajectory, so RL state resets)
- `subject_id`: the real person (for hierarchical modeling)
- `trial_id`: trial number within the trajectory (0 to 119)


In [ ]:
import json
from pathlib import Path

def load_trajectories(path):
    trajectories = []
    with open(path, 'r') as f:
        for line in f:
            line = line.strip()
            if line:
                trajectories.append(json.loads(line))
    return trajectories

def trajectories_to_dataframe(trajectories, rt_unit='ms',
                               feedback_transform='normalize',
                               group_by='trajectory', rt_placeholder=-1.0):
    records = []
    for traj in trajectories:
        ctx = traj['context']
        subj_id = int(ctx['subject_id'].split('_')[-1])
        traj_id = int(ctx['trajectory_id'].split('_')[-1])
        participant_id = subj_id if group_by == 'subject' else traj_id
        for trial in traj['trials']:
            rt = trial.get('info', {}).get('rt', None)
            if rt is None:
                rt_val = rt_placeholder
            else:
                rt_val = float(rt)
                if rt_unit == 'ms':
                    rt_val /= 1000.0
            reward = float(trial['reward'])
            feedback = reward / 100.0 if feedback_transform == 'normalize' else reward
            records.append({
                'participant_id': participant_id,
                'trial_id': trial['trial_index'],
                'response': int(trial['action']),
                'rt': rt_val,
                'feedback': feedback,
                'subject_id': ctx['subject_id'],
            })
    df = pd.DataFrame(records)
    df = df.sort_values(['participant_id', 'trial_id']).reset_index(drop=True)
    return df

def load_challenge_data(data_dir='hf_cache/public', **kwargs):
    data_dir = Path(data_dir)
    jsonl_path = data_dir / 'public_train.jsonl'
    if not jsonl_path.exists():
        from huggingface_hub import hf_hub_download
        for fname in ['public_train.jsonl', 'public_train_reward_schedules.jsonl',
                       'schema.json', 'task_description.md']:
            hf_hub_download(repo_id='mindrl-hub/mindrl-challenge-public',
                           filename=fname, repo_type='dataset',
                           local_dir=str(data_dir))
    trajectories = load_trajectories(jsonl_path)
    return trajectories_to_dataframe(trajectories, **kwargs)

df = load_challenge_data(feedback_transform='normalize',
                          rt_placeholder=-1.0, group_by='trajectory')
print(f'Loaded {len(df)} trials from {df["participant_id"].nunique()} trajectories')
print(f'Actions: {sorted(df["response"].unique())}')
print(f'Feedback range: [{df["feedback"].min():.3f}, {df["feedback"].max():.3f}]')
print(f'RT > 0: {(df["rt"] > 0).sum()} of {len(df)} trials')
df.head(8)


Each row is one trial. The person chose an arm (`response`), got a reward
(`feedback`, normalized to 0-1), and took some time (`rt`). Over 120 trials,
they learn which arm is best -- but the payout rates drift, so the best arm
changes over time.

The unfortunately named `participant_id` is used only to separate task
trajectories for HSSM. The real `subject_id` is used for hierarchical
modeling, so multiple trajectories from the same human share one random
effect while their Q values still reset at trajectory boundaries.


## 3. Model 1: Rescorla-Wagner (the simplest learner)

The Rescorla-Wagner (RW) model is the foundation of reinforcement learning
in psychology. Proposed in 1972, it says:

> After each experience, update your belief by an amount proportional to how
> surprised you were.

The formula is one line:

```
Q[chosen] += alpha * (reward - Q[chosen])
```

Where:
- `Q` is the model's estimate of each arm's quality (0 to 1)
- `alpha` is the learning rate (how fast you update, 0 to 1)
- `(reward - Q[chosen])` is the **prediction error** -- how surprised you were

If you expected 0.5 and got 0.8, PE = +0.3 (good surprise).
If you expected 0.5 and got 0.2, PE = -0.3 (bad surprise).

**Only the chosen arm is updated.** You learn nothing about arms you did not try.


In [ ]:
class NArmRescorlaWagner:
    def __init__(self, n_actions=4, initial_q=0.5, feedback_field='feedback',
                 use_decay=False):
        self._n_actions = n_actions
        self._initial_q = initial_q
        self._feedback_field = feedback_field
        self._use_decay = use_decay
        self._state = None

    @property
    def computed_params(self):
        return [f'q{i}' for i in range(self._n_actions)]

    @property
    def free_params(self):
        params = ['rl_alpha']
        if self._use_decay:
            params.append('rl_decay')
        return params

    @property
    def param_bounds(self):
        bounds = {'rl_alpha': (0.0, 1.0)}
        if self._use_decay:
            bounds['rl_decay'] = (0.0, 1.0)
        return bounds

    @property
    def default_params(self):
        defaults = {'rl_alpha': 0.2}
        if self._use_decay:
            defaults['rl_decay'] = 0.0
        return defaults

    @property
    def available_backends(self):
        return ('python', 'jax')

    @property
    def supports_gradient(self):
        return True

    @property
    def required_context_fields(self):
        return ['choice', self._feedback_field]

    def init_state(self):
        return {'q_values': np.full(self._n_actions, self._initial_q, dtype=np.float64)}

    def init_jax_state(self):
        return {'q_values': jnp.full((self._n_actions,), self._initial_q)}

    def compute_python(self, state, params, context):
        q = state['q_values']
        return {f'q{i}': float(q[i]) for i in range(self._n_actions)}

    def update_python(self, state, params, context):
        choice = int(context['choice'])
        feedback = float(context[self._feedback_field])
        alpha = params['rl_alpha']
        q = np.asarray(state['q_values'], dtype=np.float64).copy()
        if self._use_decay:
            decay = params.get('rl_decay', 0.0)
            q = (1.0 - decay) * q + decay * self._initial_q
        q[choice] += alpha * (feedback - q[choice])
        return {'q_values': q}

    def compute_jax(self, state, params, context):
        q = state['q_values']
        return {f'q{i}': q[i] for i in range(self._n_actions)}

    def update_jax(self, state, params, context):
        choice = context['choice']
        feedback = context[self._feedback_field]
        alpha = params['rl_alpha']
        q = state['q_values']
        if self._use_decay:
            decay = params.get('rl_decay', 0.0)
            q = (1.0 - decay) * q + decay * self._initial_q
        delta = feedback - q[choice]
        return {'q_values': q.at[choice].add(alpha * delta)}

print('Free params:', NArmRescorlaWagner().free_params)

# Demo: 3 trials
learner = NArmRescorlaWagner(4)
state = learner.init_state()
params = {'rl_alpha': 0.3}
print('\nTrial 0: Q =', state['q_values'])
state = learner.update_python(state, params, {'choice': 2, 'feedback': 0.8})
print('Trial 1: chose arm 2, got 0.8 -> Q =', np.round(state['q_values'], 3))
state = learner.update_python(state, params, {'choice': 2, 'feedback': 0.2})
print('Trial 2: chose arm 2, got 0.2 -> Q =', np.round(state['q_values'], 3))
print('Arm 2: 0.5 -> 0.59 (good news) -> 0.47 (bad news). Others unchanged.')


**What happened in the demo:**

1. Started with Q = [0.5, 0.5, 0.5, 0.5] (neutral beliefs)
2. Chose arm 2, got 0.8: PE = 0.8 - 0.5 = +0.3. Q[2] = 0.5 + 0.3*0.3 = 0.59
3. Chose arm 2 again, got 0.2: PE = 0.2 - 0.59 = -0.39. Q[2] = 0.59 + 0.3*(-0.39) = 0.47

The learning rate 0.3 means the model moves 30% of the way from its current
belief toward the new observation. Over many trials, Q converges to the true
average reward of each arm.

**The `@property` methods** are the contract with HSSM. HSSM reads
`free_params` to know what to estimate, `param_bounds` to constrain the search,
and `computed_params` to know what the model outputs. Think of them as labels
on a container -- HSSM reads the labels to know how to use your model.


## 4. Model 2: Dual-Alpha (asymmetric learning)

The basic RW model uses one learning rate for all surprises. But humans often
learn asymmetrically: they might quickly abandon a bad option (high rate for
negative surprises) but slowly commit to a good one (low rate for positive
surprises), or vice versa.

The dual-alpha model splits the learning rate into two:
- `rl_alpha_pos`: used when reward >= expectation (good news, positive PE)
- `rl_alpha_neg`: used when reward < expectation (bad news, negative PE)

The ONLY change from basic RW is in the update method: we check the sign of
the prediction error and pick the corresponding learning rate.


In [ ]:
class NArmDualAlphaRW:
    def __init__(self, n_actions=4, initial_q=0.5, feedback_field='feedback',
                 use_decay=False):
        self._n_actions = n_actions
        self._initial_q = initial_q
        self._feedback_field = feedback_field
        self._use_decay = use_decay
        self._state = None

    @property
    def computed_params(self):
        return [f'q{i}' for i in range(self._n_actions)]

    @property
    def free_params(self):
        params = ['rl_alpha_pos', 'rl_alpha_neg']
        if self._use_decay:
            params.append('rl_decay')
        return params

    @property
    def param_bounds(self):
        bounds = {'rl_alpha_pos': (0.0, 1.0), 'rl_alpha_neg': (0.0, 1.0)}
        if self._use_decay:
            bounds['rl_decay'] = (0.0, 1.0)
        return bounds

    @property
    def default_params(self):
        return {'rl_alpha_pos': 0.3, 'rl_alpha_neg': 0.1}

    @property
    def available_backends(self):
        return ('python', 'jax')

    @property
    def supports_gradient(self):
        return True

    @property
    def required_context_fields(self):
        return ['choice', self._feedback_field]

    def init_state(self):
        return {'q_values': np.full(self._n_actions, self._initial_q, dtype=np.float64)}

    def init_jax_state(self):
        return {'q_values': jnp.full((self._n_actions,), self._initial_q)}

    def compute_python(self, state, params, context):
        q = state['q_values']
        return {f'q{i}': float(q[i]) for i in range(self._n_actions)}

    def update_python(self, state, params, context):
        choice = int(context['choice'])
        feedback = float(context[self._feedback_field])
        a_pos = params['rl_alpha_pos']
        a_neg = params['rl_alpha_neg']
        q = np.asarray(state['q_values'], dtype=np.float64).copy()
        if self._use_decay:
            decay = params.get('rl_decay', 0.0)
            q = (1.0 - decay) * q + decay * self._initial_q
        pe = feedback - q[choice]
        alpha = a_pos if pe >= 0 else a_neg
        q[choice] += alpha * pe
        return {'q_values': q}

    def compute_jax(self, state, params, context):
        q = state['q_values']
        return {f'q{i}': q[i] for i in range(self._n_actions)}

    def update_jax(self, state, params, context):
        choice = context['choice']
        feedback = context[self._feedback_field]
        a_pos = params['rl_alpha_pos']
        a_neg = params['rl_alpha_neg']
        q = state['q_values']
        if self._use_decay:
            decay = params.get('rl_decay', 0.0)
            q = (1.0 - decay) * q + decay * self._initial_q
        pe = feedback - q[choice]
        alpha = jnp.where(pe >= 0, a_pos, a_neg)
        return {'q_values': q.at[choice].add(alpha * pe)}

print('Free params:', NArmDualAlphaRW().free_params)

# Compare basic RW vs dual-alpha
basic = NArmRescorlaWagner(4)
dual = NArmDualAlphaRW(4)
s1 = basic.init_state()
s2 = dual.init_state()
p1 = {'rl_alpha': 0.3}
p2 = {'rl_alpha_pos': 0.2, 'rl_alpha_neg': 0.7}
print('\nBasic (alpha=0.3) vs Dual (pos=0.2, neg=0.7):')
for trial, (choice, reward) in enumerate([(2, 0.8), (2, 0.2), (1, 0.6)]):
    s1 = basic.update_python(s1, p1, {'choice': choice, 'feedback': reward})
    s2 = dual.update_python(s2, p2, {'choice': choice, 'feedback': reward})
    print(f'  Trial {trial+1}: arm {choice}, reward {reward}')
    print(f'    Basic: Q = {np.round(s1["q_values"], 3)}')
    print(f'    Dual:  Q = {np.round(s2["q_values"], 3)}')


With `alpha_pos=0.2` (slow) and `alpha_neg=0.7` (fast), the dual-alpha model
reacts much more strongly to bad news. In trial 2 (reward 0.2 was disappointing),
the Q-value drops sharply with dual-alpha but only moderately with basic RW.

**When does the asymmetry matter?** Only when the prediction error is large.
Early in the task (Q starts at 0.5, rewards are 0 or 1), PEs are large and the
asymmetry has a big effect. But after Q converges toward the true reward rate,
PEs are small and the difference between alpha_pos and alpha_neg barely changes
anything. This is why dual-alpha alone barely improved predictions in the full
comparison -- by the time most trials happen, the asymmetry has already done
its job and the Q-values have converged.


## 5. Model 3: Dual-Alpha + Sticky (the submission model)

Humans have a tendency to repeat their previous choice, even when it is not
the best option. This is called **choice perseveration** or **stickiness**.
It could be motor habit, cognitive laziness, or just that your finger is
already on that button.

The `sticky` parameter adds a bonus to the last-chosen arm's Q-value, making
it more likely to be chosen again -- independent of how good it actually is.

**Key design decision:** Stickiness lives in `compute` (the output function),
not in `update` (the learning function). It is a choice bias, not a belief
update. The model's actual Q-values (its real beliefs) do NOT include the
sticky bonus -- the bonus is only added when reading out values for the decision.


In [ ]:
class NArmRWDualAlphaSticky:
    def __init__(self, n_actions=4, initial_q=0.5, feedback_field='feedback'):
        self._n_actions = n_actions
        self._initial_q = initial_q
        self._feedback_field = feedback_field
        self._state = None

    @property
    def computed_params(self):
        return [f'q{i}' for i in range(self._n_actions)]

    @property
    def free_params(self):
        return ['rl_alpha_pos', 'rl_alpha_neg', 'sticky']

    @property
    def param_bounds(self):
        return {'rl_alpha_pos': (0.0, 1.0), 'rl_alpha_neg': (0.0, 1.0),
                'sticky': (-5.0, 5.0)}

    @property
    def default_params(self):
        return {'rl_alpha_pos': 0.3, 'rl_alpha_neg': 0.1, 'sticky': 0.0}

    @property
    def available_backends(self):
        return ('python', 'jax')

    @property
    def supports_gradient(self):
        return True

    @property
    def required_context_fields(self):
        return ['choice', self._feedback_field]

    def init_state(self):
        return {'q_values': np.full(self._n_actions, self._initial_q, dtype=np.float64),
                'last_choice': -1}

    def init_jax_state(self):
        return {'q_values': jnp.full((self._n_actions,), self._initial_q),
                'last_choice': -1}

    def compute_python(self, state, params, context):
        # OUTPUT: read Q, add sticky bonus to last-chosen arm
        q = state['q_values'].copy()  # copy! don't change real beliefs
        sticky = params['sticky']
        last = state['last_choice']
        if last >= 0:
            q[last] += sticky
        return {f'q{i}': float(q[i]) for i in range(self._n_actions)}

    def update_python(self, state, params, context):
        # LEARNING: dual-alpha update + remember which arm was chosen
        choice = int(context['choice'])
        feedback = float(context[self._feedback_field])
        a_pos = params['rl_alpha_pos']
        a_neg = params['rl_alpha_neg']
        q = np.asarray(state['q_values'], dtype=np.float64).copy()
        pe = feedback - q[choice]
        alpha = a_pos if pe >= 0 else a_neg
        q[choice] += alpha * pe
        return {'q_values': q, 'last_choice': choice}

    def compute_jax(self, state, params, context):
        q = state['q_values']
        sticky = params['sticky']
        last = state['last_choice']
        q_biased = jnp.where(jnp.arange(self._n_actions) == last, q + sticky, q)
        return {f'q{i}': q_biased[i] for i in range(self._n_actions)}

    def update_jax(self, state, params, context):
        choice = context['choice']
        feedback = context[self._feedback_field]
        a_pos = params['rl_alpha_pos']
        a_neg = params['rl_alpha_neg']
        q = state['q_values']
        pe = feedback - q[choice]
        alpha = jnp.where(pe >= 0, a_pos, a_neg)
        new_q = q.at[choice].add(alpha * pe)
        return {'q_values': new_q, 'last_choice': choice}

print('Free params:', NArmRWDualAlphaSticky().free_params)

# Demo: sticky bonus in action
learner = NArmRWDualAlphaSticky(4)
state = learner.init_state()
params = {'rl_alpha_pos': 0.4, 'rl_alpha_neg': 0.7, 'sticky': 0.15}
print('\nInitial Q =', state['q_values'], 'last_choice =', state['last_choice'])
out = learner.compute_python(state, params, {})
print('Trial 0 (no previous choice): biased Q =', [round(v,3) for v in out.values()])
state = learner.update_python(state, params, {'choice': 2, 'feedback': 0.8})
out = learner.compute_python(state, params, {})
print('Trial 1 (last=arm 2): biased Q =', [round(v,3) for v in out.values()])
print(f'  Arm 2 gets +0.15 sticky: {0.59:.3f} + 0.15 = {0.74:.3f}')
print('  The softmax sees 0.74, not the raw 0.59.')


**What the demo shows:**

1. **Trial 0** (no previous choice): No sticky bonus. Biased Q = raw Q.
2. **Trial 1** (last choice was arm 2): Arm 2 gets +0.15 bonus. Even though
   arm 2's actual Q-value is 0.59, the decision module sees 0.74. This makes
   arm 2 more likely to be chosen again.

**Why stickiness is the most important feature:** Without it, the model
explains people's choice repetition by inflating beta (making the person seem
very decisive). With stickiness, beta can be moderate and stickiness handles
the repetition separately. The fitted values confirm: without sticky, beta
saturates near 10; with sticky (0.126), beta drops to 8.3.

**Positive vs negative sticky:** Bounds are [-5, 5]. Positive = repeat the
last choice (perseveration). Negative = avoid the last choice (alternation).
The fitted value of 0.126 is slightly positive -- a mild tendency to repeat.


## 6. The Decision Rule: Softmax

The learning models produce Q-values. But Q-values are not choices. We need
a rule that turns Q-values into probabilities. The standard choice is the
**softmax** with inverse temperature `beta`:

```
p(arm i) = exp(beta * Q_i) / sum_j exp(beta * Q_j)
```

- `beta = 0`: completely random (all arms equally likely)
- `beta = high`: almost always pick the highest-Q arm
- `beta = 8.3` (fitted): quite decisive, but not perfect

HSSM handles the softmax internally. But for manual NLL scoring, we write it:


In [ ]:
def softmax_logprob(q_values, beta, chosen_arm):
    scaled = beta * q_values
    max_val = jnp.max(scaled)
    exp_vals = jnp.exp(scaled - max_val)
    probs = exp_vals / jnp.sum(exp_vals)
    return jnp.log(probs[chosen_arm])

q = jnp.array([0.3, 0.5, 0.7, 0.4])
print('Q-values:', q)
for beta in [0.0, 1.0, 5.0, 8.3, 15.0]:
    scaled = beta * q
    max_val = jnp.max(scaled)
    probs = jnp.exp(scaled - max_val) / jnp.sum(jnp.exp(scaled - max_val))
    print(f'  beta={beta:5.1f}: probs = {[f"{p:.3f}" for p in probs]}')
print()
print('beta=0: random. beta=8.3: our fitted value. beta=15: almost deterministic.')


The `max_val` subtraction is a numerical stability trick: `exp(beta * Q)` can
overflow when beta and Q are both large. Subtracting the max from all exponents
keeps the largest at `exp(0) = 1.0` and prevents overflow. It does not change
the final probabilities (it cancels in the normalization).

The `beta` parameter is fitted by HSSM alongside the learning parameters. It
is part of the decision module, not the learning model.


## 7. Fitting with HSSM

Now we connect the learning models to HSSM's Bayesian fitting. Three parts:

**Part A: Priors** -- what we believe before seeing data.
**Part B: Model config** -- how to combine learner + softmax + priors.
**Part C: Fitting** -- MCMC sampling to find the best parameters.


In [ ]:
PRIOR_SPECS = {
    'rl_alpha':       (0.0, 1.0, 0.30, 0.15, 0.05),
    'rl_alpha_pos':   (0.0, 1.0, 0.30, 0.15, 0.05),
    'rl_alpha_neg':   (0.0, 1.0, 0.30, 0.15, 0.05),
    'sticky':         (-3.0, 3.0, 0.0, 0.50, 0.10),
    'beta':           (0.0, 15.0, 5.0, 2.00, 0.50),
}
print('Priors defined for:', list(PRIOR_SPECS.keys())
)

def inverse_gen_logit(value, bounds):
    lower, upper = bounds
    return lower + (upper - lower) * expit(value)

def hierarchical_param(name):
    lower, upper, mean, sd, re_sd = PRIOR_SPECS[name]
    proportion = np.clip((mean - lower) / (upper - lower), 1e-6, 1.0 - 1e-6)
    eta_mean = float(logit(proportion))
    derivative = (upper - lower) * proportion * (1.0 - proportion)
    eta_sd = float(sd / derivative)
    eta_re_sd = float(re_sd / derivative)
    return hssm.Param(
        name,
        formula=f'{name} ~ 1 + (1|subject_id)',
        bounds=(lower, upper),
        link='log_logit',
        prior={
            'Intercept': hssm.Prior('Normal', mu=eta_mean, sigma=eta_sd),
            '1|subject_id': {
                'name': 'Normal', 'mu': 0.0,
                'sigma': {'name': 'HalfNormal', 'sigma': eta_re_sd},
            },
        },
    )

print('hierarchical_param creates an HSSM Param with:')
print('  formula: param ~ 1 + (1|subject_id)')
print('  link:    log_logit (maps unconstrained to bounded)')


The formula `param ~ 1 + (1|subject_id)` means:
- Estimate a population intercept (the `1`) -- the group-average value
- Plus a random effect per real human (the `(1|subject_id)`) -- individual
  deviations from the average

Each person gets their own parameter value = intercept + their random effect.
When scoring on held-out data, we use ONLY the intercept (population average)
because held-out people are different individuals with unknown random effects.
This matches the submission agent, which uses one parameter set for everyone.

The `log_logit` link transforms bounded parameters (e.g., alpha in [0,1]) to
an unconstrained scale where the sampler can roam freely. Without it, the
sampler would hit hard walls at the boundaries.


In [ ]:
from ssms.rl import ModelConfig
from ssms.rl.env import Bandit

def make_config(description, learner, params):
    env = Bandit.bernoulli(probabilities=[0.25]*4, response_labels=[0,1,2,3])
    config = ModelConfig('inv_temp_softmax_4', description,
                         'inv_temp_softmax_4', learner, env,
                         response=['response'])
    config.bounds.update({n: PRIOR_SPECS[n][:2] for n in params})
    config.params_default = [PRIOR_SPECS[n][2] for n in config.list_params]
    config.validate()
    return config

sticky_learner = NArmRWDualAlphaSticky(4)
sticky_params = ['rl_alpha_pos', 'rl_alpha_neg', 'sticky', 'beta']
sticky_config = make_config('DualAlpha+sticky', sticky_learner, sticky_params)
print('Config: decision=softmax, learner=NArmRWDualAlphaSticky')
print(f'Params: {sticky_config.list_params}')
print(f'Bounds: {sticky_config.bounds}')


`ModelConfig` tells HSSM how to combine three pieces:
1. Your learner (produces Q-values each trial)
2. The decision module (softmax, turns Q-values into choice probabilities)
3. The parameter bounds and defaults

HSSM internally builds a full PyMC model that wires these together. For each
trial, it calls `compute_jax` to get Q-values, applies softmax, and checks how
well the probabilities match actual human choices. The MCMC sampler then
adjusts the parameters to improve the match.


In [ ]:
# Split data into training and validation with disjoint subjects
trial_counts = df.groupby('participant_id').size()
complete = trial_counts[trial_counts == 120].index
df_complete = df[df['participant_id'].isin(complete)].copy()
rng = np.random.default_rng(20260719)
subjects = np.asarray(sorted(df_complete['subject_id'].unique()))
rng.shuffle(subjects)
split_at = max(1, int(0.70 * len(subjects)))
train_subj = set(subjects[:split_at])
valid_subj = set(subjects[split_at:])
train_pool = sorted(df_complete.loc[df_complete['subject_id'].isin(train_subj), 'participant_id'].unique())
valid_pool = sorted(df_complete.loc[df_complete['subject_id'].isin(valid_subj), 'participant_id'].unique())
train_ids = rng.choice(train_pool, size=min(20, len(train_pool)), replace=False)
valid_ids = rng.choice(valid_pool, size=min(60, len(valid_pool)), replace=False)

def select(ids):
    sel = df_complete[df_complete['participant_id'].isin(ids)].copy()
    id_map = {old: new for new, old in enumerate(sorted(sel['participant_id'].unique()))}
    sel['participant_id'] = sel['participant_id'].map(id_map)
    return sel.sort_values(['participant_id', 'trial_id'])[
        ['participant_id', 'subject_id', 'trial_id', 'response', 'feedback']].reset_index(drop=True)

train_data = select(train_ids)
valid_data = select(valid_ids)
print(f'Training: {train_data["participant_id"].nunique()} trajectories')
print(f'Validation: {valid_data["participant_id"].nunique()} trajectories')
print(f'Subject overlap: 0')

# Fit the model
model_config = hssm.rl.RLSSMConfig.from_ssms_model(sticky_config)
model = hssm.RLSSM(
    data=train_data, model_config=model_config,
    p_outlier=0, lapse=None, process_initvals=True,
    include=[hierarchical_param(p) for p in sticky_params],
)
print('\nFitting (2 chains x 500 draws, ~3-5 min)...')
idata = model.sample(
    sampler='numpyro', draws=500, tune=500, chains=2, cores=1,
    target_accept=0.99, random_seed=20260719,
    progressbar=False, idata_kwargs={'log_likelihood': False},
)
print('Fitting complete!')


During fitting, HSSM:
1. Built a PyMC model combining the learner + softmax + hierarchical priors
2. For each trial, called `compute_jax` (get Q-values) and applied softmax
3. Compared predicted probabilities to actual human choices
4. The MCMC sampler (NumPyro, JAX-based) explored the parameter space
5. After 500 warm-up + 500 recorded steps per chain, returned `idata` (posterior)

The `idata` object contains the posterior samples, diagnostic information
(R-hat, ESS, divergences), and metadata about the run.


In [ ]:
import arviz as az

posterior = idata.posterior
if hasattr(posterior, 'to_dataset'):
    posterior = posterior.to_dataset()
posterior = posterior.stack(sample=('chain', 'draw'))

variables = [v for v in posterior.data_vars if v.endswith('_Intercept') or v.endswith('_sigma')]
rhat = az.rhat(posterior, var_names=variables, method='rank')
rhat_vals = np.asarray(rhat.to_array(), dtype=float)
max_rhat = float(np.nanmax(rhat_vals))
print(f'Max R-hat: {max_rhat:.4f} (want < 1.01)')

print('\nFitted parameters (population means):')
for param in sticky_params:
    eta = np.asarray(posterior[f'{param}_Intercept'].values, dtype=float)
    natural = inverse_gen_logit(eta, PRIOR_SPECS[param][:2])
    mean = float(np.mean(natural))
    q03 = float(np.quantile(natural, 0.03))
    q97 = float(np.quantile(natural, 0.97))
    print(f'  {param:20s}: {mean:.4f}  [94%: {q03:.4f}, {q97:.4f}]')

print(f'\nRandom guessing NLL: {np.log(4):.4f}')


**R-hat** tells us if the 2 chains agreed. If max R-hat < 1.01, the sampling
converged and we can trust the results.

**The fitted parameters** are the population-level means (intercepts). These
are the values that go into `config.yaml` for the submission agent. The 94%
interval tells us how confident we are.

For the submission model, the fitted values should be approximately:
- `rl_alpha_pos` ~ 0.48 (learn at moderate speed from good news)
- `rl_alpha_neg` ~ 0.75 (learn fast from bad news)
- `sticky` ~ 0.13 (mild tendency to repeat choices)
- `beta` ~ 10.2 (quite decisive)

Your exact values may differ because we used a small training set (20 subjects)
for this demo. The production run uses 100 subjects.


## 8. Scoring: How Good Are the Predictions?

After fitting, we test on **held-out data** -- people the model has never seen.
The metric is **NLL** (Negative Log-Likelihood): how surprised was the model
by the actual human choices? Lower is better.

```
NLL(trial) = -log(mean_d p_d(actual_choice))
```

We average over posterior draws: instead of using one best parameter set,
we use all plausible sets and average their predictions.


In [ ]:
def heldout_nll(idata, valid_data, learner_factory, params, bounds, n_draws=100):
    posterior = idata.posterior
    if hasattr(posterior, 'to_dataset'):
        posterior = posterior.to_dataset()
    posterior = posterior.stack(sample=('chain', 'draw'))
    rng = np.random.default_rng(20260720)
    draw_indices = rng.choice(posterior.sizes['sample'],
                              size=min(n_draws, posterior.sizes['sample']),
                              replace=False)
    theta_draws = []
    for idx in draw_indices:
        theta = {}
        for name in params:
            eta = float(np.asarray(
                posterior[f'{name}_Intercept'].isel(sample=int(idx)).values
            ).squeeze())
            theta[name] = float(inverse_gen_logit(eta, bounds[name]))
        theta_draws.append(theta)
    trajectories = [
        t.sort_values('trial_id')
        for _, t in valid_data.groupby('participant_id', sort=True)
    ]
    choices = jnp.asarray(
        np.stack([t['response'].to_numpy() for t in trajectories]), dtype=jnp.int32)
    feedback = jnp.asarray(
        np.stack([t['feedback'].to_numpy() for t in trajectories]))
    batched_theta = {
        name: jnp.asarray([t[name] for t in theta_draws])
        for name in params
    }
    learner = learner_factory()

    def score_draw(theta):
        def score_trajectory(traj_choices, traj_feedback):
            def score_trial(state, observation):
                choice, reward = observation
                computed = learner.compute_jax(state, theta, context={})
                values = jnp.stack([computed[f'q{i}'] for i in range(4)])
                log_prob = jax.nn.log_softmax(theta['beta'] * values)[choice]
                new_state = learner.update_jax(
                    state, theta,
                    context={'choice': choice, 'feedback': reward})
                return new_state, log_prob
            _, log_probs = jax.lax.scan(
                score_trial, learner.init_jax_state(),
                (traj_choices, traj_feedback))
            return log_probs
        return jax.vmap(score_trajectory)(choices, feedback)

    log_probs = np.asarray(jax.jit(jax.vmap(score_draw))(batched_theta))
    log_probs = log_probs.reshape(log_probs.shape[0], -1)
    return -logsumexp(log_probs, axis=0) + np.log(log_probs.shape[0])

sticky_bounds = {p: PRIOR_SPECS[p][:2] for p in sticky_params}
nll = heldout_nll(idata, valid_data,
                  learner_factory=lambda: NArmRWDualAlphaSticky(4),
                  params=sticky_params, bounds=sticky_bounds, n_draws=100)
print(f'Held-out NLL: {np.mean(nll):.4f} +/- {np.std(nll, ddof=1)/np.sqrt(len(nll)):.5f} SE')
print(f'Random guessing: {np.log(4):.4f}')
print(f'Model is {np.log(4)/np.mean(nll):.1f}x better than random')


**What the NLL number means:**
- `NLL = 1.386` (= log(4)): random guessing, no idea
- `NLL = 0.6`: model assigns probability exp(-0.6) = 55% to the actual choice
- `NLL = 0.0`: perfect prediction (impossible in practice)

In the production run (100 training subjects), the Sticky model achieves
NLL ~ 0.66, meaning it assigns about 52% probability to the arm the human
actually chose -- quite good for predicting noisy human behavior in a
4-choice task where random gets 25%.

**Why population-level only:** The held-out people are different from training
people. We do not have individual adjustments for them. Using the population
average matches what the submission agent does -- one parameter set for everyone.


## 9. Summary: The Model Family

We built four learning models, each adding one feature:

```
NArmRescorlaWagger        1 learning rate, no sticky
      |
      +-- NArmDualAlphaRW       2 learning rates (pos/neg), no sticky
              |
              +-- NArmRWDualAlphaSticky   2 rates + sticky (SUBMISSION MODEL)
      |
      +-- NArmRWSticky          1 learning rate + sticky (for comparison)
```

**What the production comparison found (100 train + 300 valid subjects):**

| Model | NLL | What it tells us |
|-------|-----|------------------|
| Sticky (dual+sticky) | 0.663 | Best model -- both features together |
| RW+Sticky (single+sticky) | 0.665 | Almost as good -- dual-alpha barely helps |
| HGF+Sticky | 0.670 | HGF with stickiness is close but worse |
| RW+Decay | 0.707 | Forgetting helps a little |
| RW | 0.743 | Basic model, no frills |
| HGF | 0.765 | Worst -- HGF constants may be wrong |

**Key findings:**
1. **Stickiness is the most important feature.** Adding it gives a huge improvement.
2. **Dual-alpha alone does not help.** The asymmetry is real but barely changes
   predictions because most trials have small prediction errors.
3. **Stickiness and dual-alpha together work best.** Without stickiness, beta
   inflates to absorb perseveration, distorting the learning rates.
4. **The HGF model underperforms.** The HGF constants may not be tuned for this task.

**The fitted submission parameters (production run):**
- `rl_alpha_pos = 0.48` (moderate learning from good news)
- `rl_alpha_neg = 0.75` (fast learning from bad news)
- `sticky = 0.13` (mild tendency to repeat choices)
- `beta = 10.2` (quite decisive)

These values go into `config.yaml`, and the submission agent (`agent.py`)
reads them at evaluation time. The agent never does inference -- it just
plays the model forward using these pre-fitted values.
